In [ ]:
# ============================================================
# 2) LLM CLASSIFICATION via llama-server (local OpenAI-compatible HTTP)
#    Adds columns: topic / sentiment / populism
#    + Overnight-safe: retries, checkpointing, resume, robust parallelism
# ============================================================

import json
import re
import time
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import pandas as pd


# -------------------------------
# 0) SERVER CONFIG
# -------------------------------
LLAMA_BASE_URL = "http://127.0.0.1:8080"
CHAT_URL = f"{LLAMA_BASE_URL}/v1/chat/completions"
MODELS_URL = f"{LLAMA_BASE_URL}/v1/models"

# timeout per request (seconds)
REQ_TIMEOUT = 120

# Use a Session for faster HTTP (connection reuse)
SESSION = requests.Session()

# ✅ Robust HTTP retries (helps overnight stability)
retry = Retry(
    total=5,
    backoff_factor=0.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "POST"],
    raise_on_status=False,
)
adapter = HTTPAdapter(max_retries=retry, pool_connections=50, pool_maxsize=50)
SESSION.mount("http://", adapter)
SESSION.mount("https://", adapter)


# -------------------------------
# 1) GENERATION CONFIG (speed + determinism)
# -------------------------------
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_TOKENS = 180

# Input cap (speed)
MAX_CHARS_INPUT = 2000

# Concurrency: match server parallelism (your log showed n_parallel=4)
N_WORKERS = 2

# Retries for flaky outputs
MAX_RETRIES = 2

# ✅ Control memory / stability: how many requests can be in-flight at once
MAX_IN_FLIGHT_MULT = 5  # max pending futures = N_WORKERS * this


# -------------------------------
# 1b) CHECKPOINT / RESUME CONFIG
# -------------------------------
OUTPUT_PARQUET = Path("Swissdox_articles_labeled.parquet")
OUTPUT_CSV = Path("Swissdox_articles_labeled.csv")  # optional export at end

CHECKPOINT_EVERY = 50  # save every N classified articles
PRINT_EVERY = 25        # progress print frequency


# -------------------------------
# 2) LABEL SPACE (STRICT)
# -------------------------------
TOPICS = [
    "Foreign Affairs",
    "Culture",
    "Health",
    "Social",
    "Justice",
    "Migration",
    "Defence",
    "Sport",
    "Finance",
    "Economy",
    "Education",
    "Research",
    "Environment",
    "Transports",
    "Energy",
    "Communication",
    "Other",
]
SENTIMENTS = ["POSITIVE", "NEGATIVE", "NEUTRAL"]
POPULISM = ["YES", "NO"]


# -------------------------------
# 3) PROMPT (STRICT JSON output)
# -------------------------------
SYSTEM_PROMPT = (
    "You are a strict text classification system.\n"
    "Return ONLY a valid JSON object, with no extra text.\n"
    "Follow the output schema EXACTLY.\n"
)

def build_user_prompt(title: str, subhead: str, body: str) -> str:
    return f"""
Analyze the following Swiss news article (title/subtitle/body). You must follow ALL rules.

TASKS

1) Topic recognition
Choose ONE AND ONLY ONE topic from this list:
{TOPICS}

Topic definitions:
- Foreign Affairs: international trade, cooperation, representation, diplomacy.
- Culture: culture, art, music, museums.
- Health: health, doctors, hospitals.
- Social: social affairs, elderly people, pensions, aid.
- Justice: law, courts, police, justice system.
- Migration: asylum, migration, immigration, foreigners, work permits.
- Defence: defence, army, civil protection.
- Sport: sport, clubs, promotion of sport.
- Finance: budget, taxation.
- Economy: economy, business, money, banks, trade, markets.
- Education: schools, university, high school, education, teachers.
- Research: research, innovation, science, labs, technology.
- Environment: ecology, climate change, nature protection.
- Transports: roads, trains, cars, planes, mobility.
- Energy: energy consumption, nuclear energy, electricity, oil, coal.
- Communication: TV, radio, internet networks, antennas.
- Other: not related to these topics OR mainly about another country than Switzerland.

2) Sentiment analysis TOWARDS PUBLIC ADMINISTRATION
Choose ONE label: {SENTIMENTS}
Important:
- Evaluate the tone towards public administration
- Public administration = administrative bodies/agencies/offices, civil servants, bureaucracy, administrative procedures.
- NOT public administration = parties/elected politicians in general, unless the text explicitly evaluates administrative functioning.
- If there is no clear evaluation of public administration, choose NEUTRAL.

Justification rule (STRICT):
- If sentiment_public_admin is POSITIVE or NEGATIVE, you MUST provide a short justification in sentiment_justification.
- If sentiment_public_admin is NEUTRAL, sentiment_justification MUST be an empty string: "".

3) Populism detection
Choose ONE label: {POPULISM}
Definition:
Populism rhetoric such as:
- "the people" vs "the elite/establishment" framing, OR
- anti-institution legitimacy attacks (courts, administration, "system") portrayed as corrupt/illegitimate, OR
- scapegoating an out-group framed as an enemy of "the people".
If not clear, choose NO.

Justification rule (STRICT):
- If populism is YES, you MUST provide a short justification in populism_justification.
- If populism is NO, populism_justification MUST be an empty string: "".

OUTPUT FORMAT (STRICT)
Return ONLY this JSON schema (no additional keys, no markdown):
{{
  "topic": "<one of the topics exactly as written>",
  "sentiment_public_admin": "<POSITIVE|NEGATIVE|NEUTRAL>",
  "sentiment_justification": "<string, or empty string if NEUTRAL>",
  "populism": "<YES|NO>",
  "populism_justification": "<string, or empty string if NO>"
}}

Justification style constraints:
- Max 2 sentences each.
- Must refer to concrete cues in the text (e.g., criticism/praise of authorities, bureaucracy, institutions; people-vs-elite framing).
- Do NOT add quotes longer than 12 words.

ARTICLE
TITLE: {title}
SUBTITLE: {subhead}
BODY: {body}
""".strip()


# -------------------------------
# 4) HELPERS
# -------------------------------
def clamp_text(s: str, max_chars: int) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) <= max_chars:
        return s
    head = s[: int(max_chars * 0.7)]
    tail = s[-int(max_chars * 0.3):]
    return head + " ... " + tail

def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not isinstance(text, str):
        return None
    text = text.strip()

    # Best case: the model returns a single JSON line
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("{") and line.endswith("}"):
            try:
                obj = json.loads(line)
                return obj if isinstance(obj, dict) else None
            except Exception:
                pass

    # Fallback: find the first {...} blob
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None

def normalize_and_validate(obj: Dict[str, Any]) -> Dict[str, Any]:
    topic = str(obj.get("topic", "")).strip()

    sentiment = str(obj.get("sentiment_public_admin", "")).strip().upper()
    sentiment_just = obj.get("sentiment_justification", "")
    sentiment_just = "" if sentiment_just is None else str(sentiment_just).strip()

    populism = str(obj.get("populism", "")).strip().upper()
    populism_just = obj.get("populism_justification", "")
    populism_just = "" if populism_just is None else str(populism_just).strip()

    if topic not in TOPICS:
        topic = "Other"
    if sentiment not in SENTIMENTS:
        sentiment = "NEUTRAL"
    if populism not in POPULISM:
        populism = "NO"

    # Enforce your rules even if the model messes up:
    if sentiment == "NEUTRAL":
        sentiment_just = ""
    else:
        if not sentiment_just:
            sentiment_just = ""

    if populism == "NO":
        populism_just = ""
    else:
        if not populism_just:
            populism_just = ""

    return {
        "topic": topic,
        "sentiment_public_admin": sentiment,
        "sentiment_justification": sentiment_just,
        "populism": populism,
        "populism_justification": populism_just,
    }

DEFAULT_RES = {
    "topic": "Other",
    "sentiment_public_admin": "NEUTRAL",
    "sentiment_justification": "",
    "populism": "NO",
    "populism_justification": "",
}


# -------------------------------
# 5) SERVER CALL
# -------------------------------
def get_server_model_id() -> Optional[str]:
    """Try to fetch the model id from llama-server. Not strictly required, but nice to be correct."""
    try:
        r = SESSION.get(MODELS_URL, timeout=REQ_TIMEOUT)
        r.raise_for_status()
        data = r.json()
        # OpenAI-style: {"data":[{"id":"..."}]}
        models = data.get("data", [])
        if models and isinstance(models, list):
            mid = models[0].get("id")
            return mid
    except Exception:
        pass
    return None

MODEL_ID = get_server_model_id() or "local-model"
print(f"✅ llama-server reachable, using model='{MODEL_ID}'")

def chat_completion(system_prompt: str, user_prompt: str) -> str:
    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "max_tokens": MAX_TOKENS,
    }
    r = SESSION.post(CHAT_URL, json=payload, timeout=REQ_TIMEOUT)
    r.raise_for_status()
    j = r.json()
    return j["choices"][0]["message"]["content"]

def llm_classify_one(title: str, subhead: str, content: str) -> Dict[str, Any]:
    body = clamp_text(content or "", MAX_CHARS_INPUT)
    user_prompt = build_user_prompt(title or "", subhead or "", body)

    for attempt in range(MAX_RETRIES + 1):
        try:
            text = chat_completion(SYSTEM_PROMPT, user_prompt)
            obj = extract_json_object(text)
            if obj is not None:
                return normalize_and_validate(obj)
        except Exception:
            time.sleep(0.4 * (attempt + 1))

    return dict(DEFAULT_RES)


# -------------------------------
# 6) APPLY TO df_articles (parallel + checkpoint + resume)
# -------------------------------
LABEL_COLS = [
    "topic_llm",
    "sentiment_public_admin_llm",
    "sentiment_justification_llm",
    "populism_llm",
    "populism_justification_llm",
]

def atomic_save_parquet(df: pd.DataFrame, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_parquet(tmp, index=False)
    tmp.replace(path)

def classify_dataframe(df_articles: pd.DataFrame, limit: Optional[int] = None) -> pd.DataFrame:
    # ✅ Resume if checkpoint exists
    if OUTPUT_PARQUET.exists():
        df = pd.read_parquet(OUTPUT_PARQUET)
        print(f"✅ Resuming from checkpoint: {OUTPUT_PARQUET}  shape={df.shape}")
    else:
        df = df_articles.copy()

    if limit is not None:
        df = df.head(limit).copy()

    # Ensure label columns exist
    for c in LABEL_COLS:
        if c not in df.columns:
            df[c] = pd.NA

    # Decide which rows still need classification
    pending_indices = [i for i in df.index if pd.isna(df.at[i, "topic_llm"])]
    total_pending = len(pending_indices)
    total_all = len(df)
    already_done = total_all - total_pending
    print(f"📌 Pending: {total_pending}  | Already done: {already_done}  | Total: {total_all}")

    if total_pending == 0:
        return df

    def payload_for_idx(idx) -> Tuple[str, str, str]:
        row = df.loc[idx]
        return (
            str(row.get("head", "") or ""),
            str(row.get("subhead", "") or ""),
            str(row.get("content", "") or ""),
        )

    max_in_flight = max(1, N_WORKERS * MAX_IN_FLIGHT_MULT)
    done_since_save = 0
    done_total = already_done

    def checkpoint_save():
        atomic_save_parquet(df, OUTPUT_PARQUET)
        print(f"💾 checkpoint saved -> {OUTPUT_PARQUET}")

    try:
        with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
            it = iter(pending_indices)
            future_to_idx: Dict[Any, Any] = {}

            # Prime in-flight queue
            for _ in range(min(max_in_flight, total_pending)):
                idx = next(it, None)
                if idx is None:
                    break
                title, subhead, content = payload_for_idx(idx)
                future_to_idx[ex.submit(llm_classify_one, title, subhead, content)] = idx

            while future_to_idx:
                done_set, _ = wait(future_to_idx.keys(), return_when=FIRST_COMPLETED)

                for fut in done_set:
                    idx = future_to_idx.pop(fut)

                    try:
                        res = fut.result()
                    except Exception:
                        res = dict(DEFAULT_RES)

                    # Write results
                    df.at[idx, "topic_llm"] = res["topic"]
                    df.at[idx, "sentiment_public_admin_llm"] = res["sentiment_public_admin"]
                    df.at[idx, "sentiment_justification_llm"] = res.get("sentiment_justification", "")
                    df.at[idx, "populism_llm"] = res["populism"]
                    df.at[idx, "populism_justification_llm"] = res.get("populism_justification", "")

                    done_since_save += 1
                    done_total += 1

                    # Progress
                    if done_total % PRINT_EVERY == 0 or done_total == total_all:
                        print(f"… classified {done_total}/{total_all} (session: {done_since_save}/{total_pending})")

                    # Checkpoint
                    if done_since_save % CHECKPOINT_EVERY == 0:
                        checkpoint_save()

                    # Refill one slot
                    nxt = next(it, None)
                    if nxt is not None:
                        t, sh, c = payload_for_idx(nxt)
                        future_to_idx[ex.submit(llm_classify_one, t, sh, c)] = nxt

    except KeyboardInterrupt:
        print("\n🛑 Interrupted by user. Saving checkpoint before exit…")
        checkpoint_save()
        raise
    except Exception as e:
        print(f"\n🛑 Fatal error: {e}. Saving checkpoint…")
        checkpoint_save()
        raise

    # Final save
    checkpoint_save()
    return df


# -------------------------------
# 7) RUN + SAVE
# -------------------------------
CSV_PATH = "df_articles.csv"
df_articles = pd.read_csv(CSV_PATH)

# ✅ Full overnight run: use limit=None
df_labeled = classify_dataframe(df_articles, limit=100)   # set limit=None for the real run
print("✅ df_labeled:", df_labeled.shape)
print(df_labeled[["pubtime","language","medium_code","head","topic_llm","sentiment_public_admin_llm","populism_llm"]].head(10))

# Optional exports
# df_labeled.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
# print(f"✅ Saved CSV: {OUTPUT_CSV}")


In [ ]:
import pandas as pd

df = pd.read_parquet("Swissdox_articles_labeled.parquet")

df.shape


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# 0) Prep: month column
# =========================
CSV_PATH = "2025_APERTUS_ANALYSIS.csv"
df = pd.read_csv(CSV_PATH)

df["pubtime"] = pd.to_datetime(df["pubtime"], errors="coerce")
df = df.dropna(subset=["pubtime"]).copy()
df["month"] = df["pubtime"].dt.to_period("M").dt.to_timestamp()

# Optional: restrict to year (uncomment)
# YEAR = 2025
# df = df[df["pubtime"].dt.year == YEAR].copy()

# =========================
# 1) Regroup topics -> departments
# =========================
TOPIC_TO_DEPT = {
    # EDA
    "Foreign Affairs": "EDA",

    # EDI
    "Culture": "EDI",
    "Health": "EDI",
    "Social": "EDI",

    # EJPD
    "Justice": "EJPD",
    "Migration": "EJPD",

    # VBS
    "Defence": "VBS",
    "Sport": "VBS",

    # EFD
    "Finance": "EFD",

    # WBF
    "Economy": "WBF",
    "Education": "WBF",
    "Research": "WBF",

    # UVEK
    "Environment": "UVEK",
    "Transports": "UVEK",
    "Energy": "UVEK",
    "Communication": "UVEK",

    # Other
    "Other": "Other",
}

df["dept_llm"] = df["topic_llm"].map(TOPIC_TO_DEPT).fillna("Other")

# Helper: monthly shares
def monthly_shares(df_in: pd.DataFrame, index_col: str, cat_col: str) -> pd.DataFrame:
    tab = (
        df_in.groupby([index_col, cat_col])
            .size()
            .unstack(fill_value=0)
            .sort_index()
    )
    shares = tab.div(tab.sum(axis=1).replace(0, pd.NA), axis=0).fillna(0)
    return shares

# =========================
# 2) Graph 1: Departments evolution (STACKED shares, total=1)
# =========================
dept_shares = monthly_shares(df, "month", "dept_llm")

# Optional: fix order for consistent legend / stacked layering
dept_order = ["EDA", "EDI", "EJPD", "VBS", "EFD", "WBF", "UVEK", "Other"]
for c in dept_order:
    if c not in dept_shares.columns:
        dept_shares[c] = 0
dept_shares = dept_shares[dept_order]

plt.figure(figsize=(13, 7))
plt.stackplot(
    dept_shares.index,
    dept_shares.T.values,
    labels=dept_shares.columns
)
plt.title("Evolution of departments (monthly shares, total = 1)")
plt.xlabel("Month")
plt.ylabel("Share of articles")
plt.ylim(0, 1)
plt.legend(title="Department", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# =========================
# 3) Graph 2: Sentiment evolution (shares)
# =========================
sent_shares = monthly_shares(df, "month", "sentiment_public_admin_llm")

# Ensure consistent order
sent_order = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
for c in sent_order:
    if c not in sent_shares.columns:
        sent_shares[c] = 0
sent_shares = sent_shares[sent_order]

plt.figure(figsize=(12, 6))
sent_shares.plot(ax=plt.gca())
plt.title("Sentiment towards public administration (monthly shares)")
plt.xlabel("Month")
plt.ylabel("Share of articles")
plt.ylim(0, 1)
plt.legend(title="Sentiment")
plt.tight_layout()
plt.show()

# =========================
# 4) Graph 3: Populism evolution (share of YES)
# =========================
pop_shares = monthly_shares(df, "month", "populism_llm")

# Make sure YES exists
if "YES" not in pop_shares.columns:
    pop_shares["YES"] = 0

plt.figure(figsize=(12, 6))
plt.plot(pop_shares.index, pop_shares["YES"])
plt.title("Populism over time (YES share, monthly)")
plt.xlabel("Month")
plt.ylabel("Share of articles")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# (Optional) If you prefer YES/NO stacked instead of YES-only:
# pop_order = ["NO", "YES"]
# for c in pop_order:
#     if c not in pop_shares.columns:
#         pop_shares[c] = 0
# pop_shares = pop_shares[pop_order]
# plt.figure(figsize=(13, 7))
# plt.stackplot(pop_shares.index, pop_shares.T.values, labels=pop_shares.columns)
# plt.title("Populism over time (monthly shares, total = 1)")
# plt.xlabel("Month")
# plt.ylabel("Share of articles")
# plt.ylim(0, 1)
# plt.legend(title="Populism", bbox_to_anchor=(1.02, 1), loc="upper left")
# plt.tight_layout()
# plt.show()